# BT4012 — Detecting Illicit Bitcoin Transactions

**e1156454**

This notebook reproduces the final submission end to end: loading the competition files,
building the features, validating on time-based splits, and writing the submission CSV.

## Final result

| Submission | Public LB |
|---|---|
| Gradient boosting, raw features only | 0.94351 |
| + component features | 0.94864 |
| + 2/3/4-hop propagation (`prop34_cmp`) | 0.95937 |
| PageRank propagation, α = 0.15 (`ppr_cmp`) | 0.96034 |
| Three decays mixed (`ppr_multi_cmp`) | 0.96057 |
| Three decays, 127 histogram bins | 0.96073 |
| Six decays (`ppr_fine_multi_cmp`) | 0.96066 |
| **Rank blend of all five** | **0.96137** |

## Summary of findings

The 165 supplied features already summarise each transaction's immediate neighbours, as
stated in the data description. Hand-built one-hop aggregates neighbour means, maxima,
standard deviations, therefore added nothing, and removing the one-hop propagation block
improved the score by 0.0016. The gain came from information two to four hops away in the
payment graph, or mixed across all distances with a PageRank-style decay. That single
change accounts for almost the entire rise from 0.9435 to 0.9614.

The model is unchanged throughout: scikit-learn's histogram gradient boosting with class
weighting at default settings. Four hyperparameter searches (two properly nested),
stacking, bagging, pseudo-labelling, positive-unlabelled learning, covariate-shift
weighting, nearest-neighbour label transfer, a GraphSAGE network and three other boosting
libraries were tested. Only one hyperparameter change helped: reducing histogram bins
from 255 to 127, worth 0.0002.

Runtime: 40–50 minutes, dominated by the five validation runs and five final fits.

In [ ]:
# setup
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 170)

# folder holding the four competition CSVs
DATA_DIR = Path("../data")
OUT_DIR = Path("../submissions")
OUT_DIR.mkdir(exist_ok=True)

## 1. Loading the data

Unlabelled transactions have an empty `label` field. The column is parsed explicitly so
that an unexpected value raises an error rather than being silently treated as unknown.

In [ ]:
def parse_label(raw):
    """1 = illicit, 0 = licit, blank/none/unknown = not labelled; anything else is an error."""
    text = str(raw).strip().lower()
    if text in {"", "none", "unknown", "nan"}:
        return np.nan
    if text in {"0", "0.0"}:
        return 0.0
    if text in {"1", "1.0"}:
        return 1.0
    raise ValueError(f"Unexpected label value: {raw!r}")


train = pd.read_csv(DATA_DIR / "train.csv", converters={"label": parse_label})
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
train["label"] = train["label"].astype(float)
edges.columns = ["src", "dst"]

print(f"train {train.shape}  steps {train.time_step.min()}-{train.time_step.max()}")
print(f"test  {test.shape}  steps {test.time_step.min()}-{test.time_step.max()}")
print(f"edges {len(edges):,}")
print(f"labels: {int((train.label == 0).sum()):,} licit, {int((train.label == 1).sum()):,} illicit, "
      f"{int(train.label.isna().sum()):,} unlabelled "
      f"({train.label.mean():.2%} illicit among the labelled ones)")

## 2. Inspection and cleaning decisions

The files require no cleaning: no missing or infinite feature values, no constant or
duplicated columns, no repeated transaction IDs, no train/test ID overlap. Three
decisions follow from inspection.

**Six features are excluded.** `feat_100`, `101`, `103`, `136`, `137` and `139` encode
time almost perfectly: per-step means rise monotonically with `time_step` and the train
and test distributions barely overlap. For three of them, over 99% of test values exceed
the training maximum, so a tree model cannot extrapolate and every test row falls in the
same extreme bin. The cell below prints this evidence.

**`txId` and `time_step` are not predictors.** `time_step` determines the splits only.

**Unlabelled rows are excluded from fitting and scoring** but retained in the graph,
subject to the coverage rule in section 3.

Forty-four rows across thirteen groups share identical feature vectors. No group has
conflicting labels or spans more than one time step, so all are retained: distinct
transactions can produce identical anonymised features.

In [ ]:
TIME_ENCODING = ["feat_100", "feat_101", "feat_103", "feat_136", "feat_137", "feat_139"]
RAW = [c for c in train.columns if c.startswith("feat_") and c not in TIME_ENCODING]

evidence = pd.DataFrame({
    "train_max": train[TIME_ENCODING].max(),
    "test_max": test[TIME_ENCODING].max(),
    "share_of_test_above_train_max": [(test[c] > train[c].max()).mean() for c in TIME_ENCODING],
})
print(evidence.round(3).to_string())
print(f"\nKeeping {len(RAW)} of the 165 supplied features")

## 3. Graph features

Three blocks are built from the edge list. None uses a label value.

**Coverage matching.** Test transactions are missing roughly 39% of their neighbours'
feature rows, since those transactions are absent from the files; training transactions
would otherwise have 100%. To keep the features comparable across periods, only *usable*
rows contribute: labelled training rows and test rows. This gives 63% coverage in
training against 61% in test. Two alternatives were tested, including unlabelled rows,
and relaying through featureless nodes, and both scored worse.

**Block A (components)** Size of the connected component containing each transaction, the
share of that component with usable feature rows, and degree relative to component size.
These use the complete edge list, including featureless nodes, since those still
determine component size. Every edge lies within a single time step, so components never
span periods.

**Block B (PageRank propagation)** `H ← (1−α)·P·H + α·X` for ten iterations, where `P` is
the row-normalised adjacency matrix: each transaction's features are repeatedly mixed
with the average of its neighbours', retaining a fraction α of the original at each step.
The result combines every hop distance with a geometric decay. Six decays are built
(α = 0.05 to 0.30), since mixing several outperformed any single value.

Running the propagation to convergence was tested and performed worse: 30 and 100
iterations lost 0.005–0.009 on the long-horizon splits. Ten iterations is not the
converged representation — at α = 0.05 the ten-hop term retains 0.95¹⁰ ≈ 0.6 of its
weight — but the less converged, more local version carries more signal.

**Block C (discrete hops)** `P²X`, `P³X`, `P⁴X`: features averaged over neighbourhoods at
exactly two, three and four hops. One-hop propagation is excluded; including it cost
0.0016 on the leaderboard, consistent with the supplied features already containing
one-hop statistics.

In [ ]:
def graph_setup(train, test, edges, feature_cols):
    """Combine train and test into one node table and select the usable edges.

    Returns the node table, the feature matrix, and both ends of each usable edge as row
    positions.
    """
    nodes = pd.concat([train[["txId", "time_step", "label"] + feature_cols],
                       test[["txId", "time_step"] + feature_cols].assign(label=1.0)],
                      ignore_index=True)
    # labelled training rows plus all test rows (see the coverage note above)
    usable = nodes["label"].notna().to_numpy()
    pos = pd.Series(np.arange(len(nodes)), index=nodes["txId"].values)
    e = edges[edges["src"].isin(pos.index) & edges["dst"].isin(pos.index)]
    si, di = pos.loc[e["src"]].to_numpy(), pos.loc[e["dst"]].to_numpy()
    keep = usable[si] & usable[di]
    return nodes, nodes[feature_cols].to_numpy("float32"), si[keep], di[keep]


def transition_matrix(si, di, n):
    """Row-normalised adjacency, treated as undirected. Row i averages over i's neighbours."""
    rows, cols = np.r_[si, di], np.r_[di, si]
    A = sparse.coo_matrix((np.ones(len(rows), "float32"), (rows, cols)), shape=(n, n)).tocsr()
    deg = np.asarray(A.sum(axis=1)).ravel()
    inv = np.zeros_like(deg)
    inv[deg > 0] = 1.0 / deg[deg > 0]      # isolated nodes keep a zero row
    return sparse.diags(inv.astype("float32")) @ A, deg


nodes, X, SI, DI = graph_setup(train, test, edges, RAW)
P, DEG = transition_matrix(SI, DI, len(nodes))
IDX = pd.Index(nodes["txId"].values, name="txId")
print(f"{len(nodes):,} nodes, {len(SI):,} usable edges, "
      f"{(DEG == 0).sum():,} nodes with no usable neighbour")

In [ ]:
# --- Block A: component features ---------------------------------------------
# Union-find over the complete edge list. The node IDs are not contiguous, so this is
# simpler than relabelling them for scipy.sparse.csgraph.
def component_features(edges, node_ids, usable_ids):
    all_ids = pd.unique(pd.concat([edges["src"], edges["dst"], pd.Series(node_ids)]))
    pos = pd.Series(np.arange(len(all_ids)), index=all_ids)
    parent = np.arange(len(all_ids))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]   # path halving
            a = parent[a]
        return a

    for u, v in zip(pos.loc[edges["src"]].to_numpy(), pos.loc[edges["dst"]].to_numpy()):
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[ru] = rv
    root = np.array([find(i) for i in range(len(all_ids))])

    df = pd.DataFrame({"root": root, "usable": pd.Index(all_ids).isin(usable_ids)})
    size = df.groupby("root")["usable"].size()
    share = df.groupby("root")["usable"].mean()
    degree = pd.concat([edges["src"], edges["dst"]]).value_counts()

    rows = pos.loc[node_ids].to_numpy()
    out = pd.DataFrame(index=pd.Index(node_ids, name="txId"))
    out["cmp_size"] = df["root"].map(size).to_numpy()[rows]
    out["cmp_usable_share"] = df["root"].map(share).to_numpy()[rows]
    out["cmp_degree_over_size"] = (pd.Index(node_ids).map(degree).fillna(0).to_numpy()
                                   / out["cmp_size"].to_numpy())
    return out.astype("float32")


usable_ids = set(nodes.loc[nodes["label"].notna(), "txId"])
cmp_feats = component_features(edges, nodes["txId"].to_numpy(), usable_ids)
print(cmp_feats.describe().round(3).to_string())

In [ ]:
# --- Blocks B and C: propagation ---------------------------------------------
def ppr_features(X, P, deg, index, cols, alpha, prefix, iters=10):
    """PageRank-style propagation. Ten iterations: longer runs scored worse (section 3)."""
    H = X.copy()
    for _ in range(iters):
        H = (1 - alpha) * (P @ H) + alpha * X
    out = pd.DataFrame(H, index=index, columns=[f"{prefix}{c}" for c in cols])
    out[deg == 0] = np.nan
    return out.astype("float32")


def hop_features(X, P, deg, index, cols, hops=(2, 3, 4)):
    """Averages over neighbourhoods at exactly 2, 3 and 4 hops; 1-hop is excluded."""
    blocks, H = {}, X
    for hop in range(1, max(hops) + 1):
        H = P @ H
        if hop in hops:
            frame = pd.DataFrame(H, index=index, columns=[f"p{hop}_{c}" for c in cols])
            frame[deg == 0] = np.nan
            blocks[hop] = frame.astype("float32")
    return pd.concat(blocks.values(), axis=1)


ALPHAS = {"ppr_": 0.15, "ppr05_": 0.05, "ppr30_": 0.30,
          "pprf10_": 0.10, "pprf20_": 0.20, "pprf25_": 0.25}
ppr_blocks = {prefix: ppr_features(X, P, DEG, IDX, RAW, alpha=a, prefix=prefix)
              for prefix, a in ALPHAS.items()}
hops = hop_features(X, P, DEG, IDX, RAW)
print(f"{len(ppr_blocks)} PageRank blocks, {ppr_blocks['ppr_'].shape[1]} columns each")
print(f"hop block: {hops.shape[1]} columns")
print(f"missing rate across the propagated columns: {ppr_blocks['ppr_'].isna().mean().mean():.1%}")

In [ ]:
# --- the four feature sets the five models use -------------------------------
graph_block = pd.concat([cmp_feats] + list(ppr_blocks.values()) + [hops], axis=1)
train_f = train.join(graph_block, on="txId")
test_f = test.join(graph_block, on="txId")

CMP = list(cmp_feats.columns)
cols_of = lambda prefix: list(ppr_blocks[prefix].columns)
FEATURES = {
    "ppr_cmp":            RAW + cols_of("ppr_") + CMP,
    "prop34_cmp":         RAW + list(hops.columns) + CMP,
    "ppr_multi_cmp":      RAW + cols_of("ppr_") + cols_of("ppr05_") + cols_of("ppr30_") + CMP,
    "ppr_fine_multi_cmp": RAW + cols_of("ppr_") + cols_of("ppr05_") + cols_of("ppr30_")
                              + cols_of("pprf10_") + cols_of("pprf20_") + cols_of("pprf25_") + CMP,
}
for name, cols in FEATURES.items():
    print(f"{name:20s} {len(cols):5d} features")

## 4. Validation design

Training covers time steps 1–35 and test covers 36–49, so validation must predict later
periods from earlier ones. Shuffled k-fold would expose future information and
substantially overstate performance.

| Split | Training steps | Validation steps |
|---|---|---|
| fold1–fold4 | 1–15, 1–20, 1–25, 1–30 | the following 5 steps |
| horizon | 1–21 | 22–35 (14 steps, matching the test gap) |

Unlabelled rows are excluded from fitting and scoring. Because every edge lies within one
time step, a time-based split cannot separate the two ends of an edge.

This scheme reliably identified poor models — a GraphSAGE network scored 0.84 here and
0.88 on the leaderboard — but could not rank close candidates. Across nine probe
submissions the rank correlation between validation score and public leaderboard was
0.00, and six configurations that improved every validation metric subsequently lost on
test. No split can observe steps 36–49, which is the most likely explanation.

In [ ]:
SPLITS = [("fold1", (1, 15), (16, 20)), ("fold2", (1, 20), (21, 25)),
          ("fold3", (1, 25), (26, 30)), ("fold4", (1, 30), (31, 35)),
          ("horizon", (1, 21), (22, 35))]


def make_model(max_bins=255):
    """The final model. Early stopping is disabled so the number of trees is fixed and the
    model refitted on all labelled rows is exactly the one that was validated."""
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=20,
        l2_regularization=0.0, max_bins=max_bins, early_stopping=False,
        class_weight="balanced", random_state=SEED)


def validate(frame, feature_cols, max_bins=255, splits=SPLITS):
    labelled = frame[frame["label"].notna()]
    rows = []
    for name, (tr_lo, tr_hi), (va_lo, va_hi) in splits:
        tr = labelled[labelled["time_step"].between(tr_lo, tr_hi)]
        va = labelled[labelled["time_step"].between(va_lo, va_hi)]
        model = make_model(max_bins).fit(tr[feature_cols], tr["label"].astype(int))
        p = model.predict_proba(va[feature_cols])[:, 1]
        rows.append({"split": name, "auc": roc_auc_score(va["label"], p),
                     "ap": average_precision_score(va["label"], p),
                     "n_val": len(va), "n_illicit": int(va["label"].sum())})
    res = pd.DataFrame(rows).set_index("split")
    folds = [s[0] for s in splits if s[0].startswith("fold")]
    res.loc["fold mean"] = [res.loc[folds, "auc"].mean(), res.loc[folds, "ap"].mean(),
                            np.nan, np.nan]
    return res

In [ ]:
# The five models: four feature sets at the default 255 bins, plus ppr_multi_cmp at 127
# bins, the only hyperparameter change that improved the score
MODELS = [("ppr_cmp", "ppr_cmp", 255),
          ("prop34_cmp", "prop34_cmp", 255),
          ("ppr_multi_cmp", "ppr_multi_cmp", 255),
          ("ppr_fine_multi_cmp", "ppr_fine_multi_cmp", 255),
          ("ppr_multi_cmp_bins127", "ppr_multi_cmp", 127)]

In [ ]:
validation = {}
for label, fs, bins in MODELS:
    print(f"\nValidating {label} ({len(FEATURES[fs])} features, {bins} bins) ...")
    validation[label] = validate(train_f, FEATURES[fs], max_bins=bins)
    print(validation[label].round(4).to_string())

summary = pd.DataFrame({label: res["auc"] for label, res in validation.items()}).T
print("\n=== AUC by split, all five models ===")
print(summary.round(4).to_string())

## 5. Results

| Feature set | Fold mean AUC | Public LB |
|---|---|---|
| raw features only (159) | 0.9830 | 0.94351 |
| + component features | 0.9826 | 0.94864 |
| + 1/2-hop propagation | 0.9821 | 0.95670 |
| 2/3/4-hop propagation | 0.9865 | 0.95937 |
| PageRank, α = 0.15 | 0.9865 | 0.96034 |
| three decays | 0.9859 | 0.96057 |
| six decays | 0.9863 | 0.96066 |
| three decays, 127 bins | 0.9854 | 0.96073 |
| **rank blend of five models** | — | **0.96137** |

Two observations shaped the final approach.

**Validation and leaderboard scores were largely unrelated.** Adding 1/2-hop propagation
lowered the fold mean and raised the leaderboard score by 0.008. This pattern recurred
throughout. Validation was therefore used as a filter for clearly poor configurations
rather than as a ranking, with submissions spent on a list of candidates fixed in
advance.

**Blending succeeded under one specific condition:** members within approximately 0.001
of each other on the leaderboard that summarise the graph differently. Under those
conditions the blend beat all its members in five out of five attempts; when a member was
substantially weaker, or two members shared a representation, the blend landed between
them or below. The five models here differ in decay structure, hop structure and binning,
with rank correlations around 0.96–0.98.

Approaches tested and rejected: hand-built one-hop aggregates, degree features, per-step
rank normalisation, dropping drift-prone features, restricted training windows, recency
weighting, stacking, bagging, pseudo-labelling, positive-unlabelled learning,
covariate-shift weighting, nearest-neighbour label transfer, score smoothing over the
graph and in feature space, directed propagation, max-propagation, symmetric
normalisation, contrast features, a two-stage propagated-score model, a GraphSAGE
network, and LightGBM, XGBoost, CatBoost, random forests and extra trees.

## 6. Final model and submission

Each of the five models is refitted on all labelled training rows (steps 1–35, unlabelled
excluded) with the settings it was validated under, then combined by average rank. Ranks
rather than probabilities, because AUC depends only on ordering and the models are
calibrated differently.

The nested weights below reproduce the order in which the blends were built and scored on
Kaggle.

In [ ]:
def fit_predict_full(frame_tr, frame_te, feature_cols, max_bins=255):
    labelled = frame_tr[frame_tr["label"].notna()]
    model = make_model(max_bins).fit(labelled[feature_cols], labelled["label"].astype(int))
    return model.predict_proba(frame_te[feature_cols])[:, 1]


def rank01(p):
    """Ranks scaled into (0, 1); tied predictions share a rank."""
    return rankdata(p) / (len(p) + 1)


preds = {}
for label, fs, bins in MODELS:
    preds[label] = fit_predict_full(train_f, test_f, FEATURES[fs], max_bins=bins)
    print(f"{label:24s} mean {preds[label].mean():.4f}, median {np.median(preds[label]):.4f}")

# Combined in the order the blends were built and scored:
pair        = 0.5  * rank01(preds["ppr_cmp"])  + 0.5  * rank01(preds["prop34_cmp"])
with_multi  = 2/3  * rank01(pair)              + 1/3  * rank01(preds["ppr_multi_cmp"])
with_fine   = 0.75 * rank01(with_multi)        + 0.25 * rank01(preds["ppr_fine_multi_cmp"])
final       = 0.75 * rank01(with_fine)         + 0.25 * rank01(preds["ppr_multi_cmp_bins127"])

print("\nRank correlation between the five models (lower = more diverse):")
print(pd.DataFrame({k: rank01(v) for k, v in preds.items()}).corr(method="spearman").round(4).to_string())

In [ ]:
# --- write the submission, with format checks -------------------------------
submission = pd.DataFrame({"index": test["index"].to_numpy(), "target": final})

assert len(submission) == len(sample), "row count does not match sample_submission"
assert submission["index"].equals(sample["index"]), "index does not match sample_submission"
assert np.isfinite(submission["target"]).all(), "non-finite predictions"
assert submission["target"].between(0, 1).all(), "predictions outside [0, 1]"
assert list(submission.columns) == ["index", "target"], "unexpected columns"

path = OUT_DIR / "e1156454_submission.csv"
submission.to_csv(path, index=False)
print(f"All checks passed. Wrote {path} ({len(submission):,} rows)")
submission.head()

## 7. Conclusion

The model is histogram gradient boosting with class weighting at default settings, used
unchanged from the third submission to the last. The improvement came entirely from
feature representation rather than from the learner.

The supplied features already encode each transaction's immediate neighbourhood, so
one-hop features were redundant. The remaining signal sat two to four hops away in the
payment graph, which is consistent with laundering patterns that move funds through
chains of intermediaries: the evidence identifying a transaction may be several steps
removed from it.

The principal limitation is that chronological validation could not distinguish close
candidates. Six configurations improved every validation metric and then lost on test,
and the rank correlation across nine probe submissions was 0.00. No split can observe the
period being scored. A deployed anti-money-laundering system would face the same problem
without a leaderboard to detect it; addressing it would require labels from the scored
period, which this setting withholds.

*Reproducibility.* This notebook is a standalone reimplementation of the staged pipeline
in `src/` and does not import from it. The submission it produces scored 0.96137 on the
public leaderboard, matching the file built through the staged code.